# Gaming Behaviour and Academic Performance Analysis
**B.Tech AI & Data Science | Beginner project**

## Objective
Explore how daily gaming time is associated with academic scores while considering study hours, sleep, attendance, and stress.

> **Data honesty:** This notebook uses synthetic (simulated) data for practice. It is not real student evidence. The analysis is descriptive and cannot prove cause and effect.


## 1. Import libraries and load the dataset
Upload `gaming_academic_synthetic_data.csv` using the Colab Files panel, or run the upload cell below.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")


In [ ]:
# If the CSV is not already in the Colab session, uncomment and run:
# from google.colab import files
# uploaded = files.upload()

DATA_PATH = "gaming_academic_synthetic_data.csv"
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())


## 2. Understand the data

In [ ]:
print("Column types and non-null counts:")
df.info()

print("\nMissing values:")
display(df.isnull().sum().to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())
display(df.describe().round(2))


## 3. Clean the data
For this beginner example, missing numeric values are replaced with each column's median. In a real project, explain and justify the chosen method.

In [ ]:
df_clean = df.copy()
numeric_cols = df_clean.select_dtypes(include="number").columns
df_clean[numeric_cols] = df_clean[numeric_cols].apply(pd.to_numeric, errors="coerce")
df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].median())
df_clean = df_clean.drop_duplicates()

# Basic range checks
df_clean["daily_gaming_hours"] = df_clean["daily_gaming_hours"].clip(0, 24)
df_clean["daily_study_hours"] = df_clean["daily_study_hours"].clip(0, 24)
df_clean["sleep_hours"] = df_clean["sleep_hours"].clip(0, 24)
df_clean["attendance_percent"] = df_clean["attendance_percent"].clip(0, 100)
df_clean["academic_score_percent"] = df_clean["academic_score_percent"].clip(0, 100)

print("Missing values after cleaning:")
display(df_clean.isnull().sum().to_frame("missing_count"))
print("Cleaned shape:", df_clean.shape)


## 4. Group students by daily gaming time

In [ ]:
bins = [-0.01, 1, 3, 5, 24]
labels = ["0–1 hour", ">1–3 hours", ">3–5 hours", ">5 hours"]
df_clean["gaming_group"] = pd.cut(df_clean["daily_gaming_hours"], bins=bins, labels=labels)

group_summary = df_clean.groupby("gaming_group", observed=False).agg(
    students=("student_id", "count"),
    mean_score=("academic_score_percent", "mean"),
    mean_study_hours=("daily_study_hours", "mean"),
    mean_sleep_hours=("sleep_hours", "mean")
).round(2)
display(group_summary)


## 5. Visualize gaming hours and academic score

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_clean, x="daily_gaming_hours", y="academic_score_percent", alpha=0.7)
sns.regplot(data=df_clean, x="daily_gaming_hours", y="academic_score_percent",
            scatter=False, color="red")
plt.title("Daily Gaming Hours vs Academic Score")
plt.xlabel("Gaming hours per day")
plt.ylabel("Academic score (%)")
plt.tight_layout()
plt.show()


## 6. Compare average scores across gaming groups

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=df_clean, x="gaming_group", y="academic_score_percent", errorbar=None)
plt.title("Average Academic Score by Gaming Group")
plt.xlabel("Daily gaming group")
plt.ylabel("Average academic score (%)")
plt.tight_layout()
plt.show()


## 7. Check study hours and academic score

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_clean, x="daily_study_hours", y="academic_score_percent",
                hue="gaming_group", alpha=0.75)
plt.title("Study Hours vs Academic Score")
plt.xlabel("Study hours per day")
plt.ylabel("Academic score (%)")
plt.tight_layout()
plt.show()


## 8. Correlation heatmap

In [ ]:
corr_cols = ["daily_gaming_hours", "daily_study_hours", "sleep_hours",
             "attendance_percent", "stress_level_1_to_10", "academic_score_percent"]
corr = df_clean[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

print("Correlation with academic score (descriptive only):")
display(corr["academic_score_percent"].drop("academic_score_percent").sort_values(ascending=False).round(3))


## 9. Findings and interpretation
Write 3–5 observations based on the actual charts and values generated above. Do not claim that gaming causes lower or higher grades based on correlation alone.

In [ ]:
print("Number of records:", len(df_clean))
print("Average gaming hours:", round(df_clean["daily_gaming_hours"].mean(), 2))
print("Average study hours:", round(df_clean["daily_study_hours"].mean(), 2))
print("Average sleep hours:", round(df_clean["sleep_hours"].mean(), 2))
print("Average academic score:", round(df_clean["academic_score_percent"].mean(), 2))

print("\nInterpretation checklist:")
print("1. Compare average scores across gaming groups.")
print("2. Look at the gaming-vs-score scatter plot and correlation.")
print("3. Check whether study hours and sleep also vary.")
print("4. Mention that the dataset is synthetic and correlations do not prove causation.")


## 10. Limitations and future improvements
- The dataset is synthetic, not collected from real students.
- Simulated relationships may not reflect real-world patterns.
- Correlation does not establish cause and effect.
- Real data may include measurement errors and confounding factors.
- For a real study, collect anonymous data with consent, protect privacy, document the sampling method, and cite the dataset source.
- Future work: use a verified public dataset or an approved anonymous survey, then compare results.

## Conclusion
This project demonstrates a basic data science workflow: data loading, inspection, cleaning, group analysis, visualization, and interpretation. Replace synthetic data with an approved real dataset before making claims about actual students.
